# Cooling Scenario Workflow for Top 1000 High-Exposure Street Segments

This notebook generates cooling scenarios for the top 1000 high-exposure street segments identified from the baseline UTCI analysis. For each cooling level, the workflow modifies hourly street-level UTCI GeoPackages by subtracting a fixed cooling value from the selected high-exposure segments. The cooled networks are then used in the downstream routing and exposure workflow to recompute trip-level UTCI exposure and segment-level route counts.

The same workflow was also repeated for top 150 and top 400 intervention scenarios, but those repeated blocks are omitted from this notebook to keep the workflow focused and reproducible.

## 3°C Cooling Scenario

This step reads the top 1000 high-exposure segment file, extracts the seg_id values, and loops through the 24 baseline hourly street-level UTCI GeoPackages. For each hourly file, the notebook subtracts 3°C from the utci value of selected top-1000 segments and writes the modified GeoPackage to the 3°C cooling scenario folder.

In [8]:
import geopandas as gpd
from pathlib import Path

# 1) Update paths for Top-1000 and the new “top1000” scenario folder
top_segments_fp = Path(
    r"C:\Users\Agustin\Documents\2025summer\sanity_check\results\temp_segments\top1000_high_utci_segments_neat_GLOBAL.gpkg"
)
base_dir   = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\results")
output_dir = Path(
    r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results"
)

# 2) Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# 3) Load Top-1000 segment IDs
top_gdf = gpd.read_file(top_segments_fp)
top_ids = set(top_gdf["seg_id"].astype(int))

# 4) Process each hourly file
for hh in range(24):
    hour   = f"{hh:02d}"
    base_fp = base_dir / f"street_segments_utci_neat_{hour}.gpkg"
    out_fp  = output_dir  / f"street_segments_utci_neat_{hour}.gpkg"

    try:
        gdf = gpd.read_file(base_fp)
    except Exception as e:
        print(f"Skipping hour {hour}: cannot read {base_fp} ({e})")
        continue

    # Subtract 3°C for Top-1000 segments
    mask = gdf["seg_id"].isin(top_ids)
    gdf.loc[mask, "utci"] = gdf.loc[mask, "utci"] - 3

    # Save modified layer
    gdf.to_file(out_fp, driver="GPKG")
    print(f"Processed hour {hour}, wrote {out_fp}")

print("3 °C cooling scenario completed for Top-1000 segments.")


Processed hour 00, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_00.gpkg
Processed hour 01, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_01.gpkg
Processed hour 02, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_02.gpkg
Processed hour 03, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_03.gpkg
Processed hour 04, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_04.gpkg
Processed hour 05, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_segments_utci_neat_05.gpkg
Processed hour 06, wrote C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scen

## CELL X - RUN PYTHON SCRIPT

#### The cooled street networks generated above are used as inputs to a separate analysis script that recomputes trip-level UTCI exposure and segment-level usage counts.

This step is executed outside of this notebook using the script:

#### cooling_scenario_analysis.py

This script performs the following operations:

Loads routed Citi Bike trips (merged_routes_neat_*.gpkg),
Loads cooled street-level UTCI segment files,
Intersects routes with cooled segments,
Computes per-trip mean UTCI exposure,
Counts usage of high-UTCI segments

Outputs:
mean_utci_*.csv (trip-level exposure)
street_network_with_counts.gpkg (segment-level counts)

To run this step, execute the following command from the repository root:
#### python scripts/cooling_scenario_analysis.py

Ensure that all input and output paths inside the script are configured relative to the repository structure.

The cooled trip-level CSV files may contain missing trip_len_m values after rerunning the exposure workflow. Because route geometry and trip distance do not change between the baseline and cooling scenarios, this step copies trip_len_m from the baseline CSV files using ride_id as the join key.

In [1]:
import pandas as pd
from pathlib import Path

# Directories
base_dir = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\utci_results")
cool_dir = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\utci_results")

# Loop over each hourly file
for hh in range(24):
    hour_str = f"{hh:02d}00"
    base_fp = base_dir / f"mean_utci_{hour_str}.csv"
    cool_fp = cool_dir / f"mean_utci_{hour_str}.csv"
    
    if not base_fp.exists() or not cool_fp.exists():
        print(f"Skipping {hour_str}: file missing in base or cooled directory.")
        continue
    
    # Load base lengths and cooled data
    df_base = pd.read_csv(
        base_fp,
        usecols=["ride_id", "trip_len_m"]
    ).rename(columns={"trip_len_m": "trip_len_m_base"})
    df_cool = pd.read_csv(cool_fp)
    
    # Merge and restore original lengths
    df_merged = df_cool.merge(df_base, on="ride_id", how="left")
    df_merged["trip_len_m"] = df_merged["trip_len_m_base"]
    
    # Reorder columns to match original
    df_merged = df_merged[df_cool.columns]
    
    # Overwrite the cooled CSV
    df_merged.to_csv(cool_fp, index=False)
    print(f"✔ Updated trip_len_m in cooled CSV for {hour_str}")


✔ Updated trip_len_m in cooled CSV for 0000
✔ Updated trip_len_m in cooled CSV for 0100
✔ Updated trip_len_m in cooled CSV for 0200
✔ Updated trip_len_m in cooled CSV for 0300
✔ Updated trip_len_m in cooled CSV for 0400
✔ Updated trip_len_m in cooled CSV for 0500
✔ Updated trip_len_m in cooled CSV for 0600
✔ Updated trip_len_m in cooled CSV for 0700
✔ Updated trip_len_m in cooled CSV for 0800
✔ Updated trip_len_m in cooled CSV for 0900
✔ Updated trip_len_m in cooled CSV for 1000
✔ Updated trip_len_m in cooled CSV for 1100
✔ Updated trip_len_m in cooled CSV for 1200
✔ Updated trip_len_m in cooled CSV for 1300
✔ Updated trip_len_m in cooled CSV for 1400
✔ Updated trip_len_m in cooled CSV for 1500
✔ Updated trip_len_m in cooled CSV for 1600
✔ Updated trip_len_m in cooled CSV for 1700
✔ Updated trip_len_m in cooled CSV for 1800
✔ Updated trip_len_m in cooled CSV for 1900
✔ Updated trip_len_m in cooled CSV for 2000
✔ Updated trip_len_m in cooled CSV for 2100
✔ Updated trip_len_m in cooled C

CALCULATE LENGTH OF SEGMENTS IN "C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_network_with_counts.gpkg"

Wrote: C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\3degrees_top1000_street_network_with_counts_and_length.gpkg

In [4]:
import geopandas as gpd
from pathlib import Path

# === USER-ADJUST PATHS FOR TOP-1000, 3 °C SCENARIO ===
INPUT_FP  = Path(
    r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\street_network_with_counts.gpkg"
)
OUTPUT_FP = INPUT_FP.parent / "3degrees_top1000_street_network_with_counts_and_length.gpkg"

# Load and reproject to EPSG:32118 (so lengths are in meters)
gdf = gpd.read_file(INPUT_FP).to_crs(32118)

# Compute segment length in meters
gdf["length_m"] = gdf.geometry.length

# Drop the 'utci' column if present
if "utci" in gdf.columns:
    gdf = gdf.drop(columns=["utci"])

# Select and order columns: seg_id, count, length_m, geometry
cols = ["seg_id", "count", "length_m", "geometry"]
gdf = gdf[cols]

# Save new GeoPackage
gdf.to_file(OUTPUT_FP, driver="GPKG")
print("Wrote:", OUTPUT_FP)


Wrote: C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\3degrees_top1000_street_network_with_counts_and_length.gpkg


END 3 DEGREES SCENARIO

### This notebook demonstrates the workflow for the 3°C cooling scenario applied to the top 1000 high-exposure street segments.

### The same workflow was executed for 5°C and 10°C cooling scenarios using identical logic, with only the cooling magnitude changed. These additional scenarios are not shown here to avoid redundancy but are included in the repository outputs.